# Waymo PUDO risk: data structure, preprocessing, likelihood, and consequence

This notebook follows the Fall 2026 risk exercise in its intended order:

1. define the hazard and the event being counted;
2. characterize consequence without treating a complaint as an injury;
3. estimate likelihood while controlling for exposure;
4. test whether exposure-normalized frequency is rising or falling;
5. place explicitly defined scenarios on a provisional risk matrix.

The core distinction is between **PUDO complaints**, **PUDO collisions**, and
**injury-coded PUDO collisions**. They are related risk signals, not
interchangeable labels for one outcome.


In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import numpy as np
import pandas as pd
import polars as pl
from scipy import stats
import statsmodels.api as sm

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

DATA = Path(os.environ.get("PUDO_DATA_DIR", "../data")) / "parquet"
paths = {
    "summary": DATA / "pudo_summary.parquet",
    "monthly": DATA / "monthly_activity.parquet",
    "events": DATA / "complaints.parquet",
    "locations": DATA / "collision_locations.parquet",
}
missing = [str(path) for path in paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Run pudo-build first; missing: {missing}")

summary = pl.read_parquet(paths["summary"])
monthly = pl.read_parquet(paths["monthly"])
events = pl.read_parquet(paths["events"])
locations = pl.read_parquet(paths["locations"])


In [ ]:
required = [
    "pudo_complaints",
    "pudo_collisions",
    "pudo_collisions_incident_report",
    "pudo_collisions_location",
    "pudo_collision_reconciliation_gap",
    "vmt",
    "trips",
    "pudo_per_100k_vmt",
    "pudo_per_100k_trips",
    "time_months",
]
missing_columns = sorted(set(required) - set(summary.columns))
if missing_columns:
    raise ValueError(f"Summary is stale; missing columns: {missing_columns}")
null_counts = summary.select(
    [pl.col(column).null_count().alias(column) for column in required]
).row(0)
bad = {column: count for column, count in zip(required, null_counts) if count}
if bad:
    raise ValueError(f"Summary contains null critical values: {bad}")
if summary.filter((pl.col("vmt") <= 0) | (pl.col("trips") <= 0)).height:
    raise ValueError("Summary contains non-positive exposure")
if summary.get_column("period_id").n_unique() != summary.height:
    raise ValueError("Summary contains duplicate reporting periods")

s = summary.sort("period_midpoint").to_pandas()
s["schema_era"] = np.where(
    s["period_id"].str.startswith("2024"), "2024 aggregate", "2025+ row-level"
)
post = s[s["schema_era"] == "2025+ row-level"].copy()

worst_columns = [
    "pudo_collision_worst_property_damage",
    "pudo_collision_worst_injury_possible",
    "pudo_collision_worst_injury_minor",
    "pudo_collision_worst_injury_severe",
    "pudo_collision_worst_fatality",
    "pudo_collision_worst_no_listed_severity",
]
if not np.allclose(
    post[worst_columns].sum(axis=1), post["pudo_collisions_incident_report"]
):
    raise ValueError("Mutually exclusive severity counts do not reconcile")
if not (post["pudo_collision_primary_source"] == "Incidents-Location").all():
    raise ValueError("Modern collision frequency is not using Incidents-Location")

print(
    f"Validated {len(s)} reporting windows, {len(post)} comparable modern windows, "
    f"{len(monthly)} months, and {len(events):,} retained analytic rows."
)


## 1. What is a sample, and what is a feature?

This project has several grains. They must not be mixed into one machine-
learning table. In particular, an event row has no row-level VMT or trip
exposure and no join key linking it to a specific month. The inferential sample
for the primary trend is therefore six reporting windows—not millions of raw
CSV rows.


In [ ]:
modern_event_rows = events.filter(
    pl.col("period_id").str.starts_with("2025")
    | pl.col("period_id").str.starts_with("2026")
).height
modern_location_rows = locations.filter(
    pl.col("period_id").str.starts_with("2025")
    | pl.col("period_id").str.starts_with("2026")
).height

sample_table = pd.DataFrame(
    [
        ["Monthly activity", len(monthly), "month", "Exposure/operations; descriptive trend"],
        ["Actual CPUC windows", len(s), "reporting window", "Full-window description; schema-confounded"],
        ["Comparable 2025+ windows", len(post), "reporting window", "Primary count-rate models"],
        ["Retained 2025+ analytic events", modern_event_rows, "flagged event row", "Category, actor, and severity cross-tabs"],
        ["2025+ location rows", modern_location_rows, "census tract/window", "Collision numerator reconciliation only"],
    ],
    columns=["data layer", "n", "grain", "valid use"],
)

source_archive_audit = pd.DataFrame(
    [
        ["Exact duplicate all-N/redacted template rows", 23_825_756, "stream-filter; never treat as negative samples"],
        ["Non-filler modern source rows", 11_608, "raw audit population, including citations/unclassified rows"],
        ["PUDO complaints", 325, "period-level outcome count"],
        ["PUDO collision actor rows", 315, "severity/actor linkage only"],
    ],
    columns=["source-ZIP audit item", "count", "handling"],
)
display(sample_table)
display(source_archive_audit)


In [ ]:
feature_map = pd.DataFrame(
    [
        ["Monthly", "Trips, waiting, P1/P2/P3 VMT, passengers, PMT", "operating scale and composition"],
        ["Window", "time, duration, complaint/collision counts, VMT, trips", "count outcome + one exposure offset"],
        ["Event", "complaint flags, collision actors, severity flags", "categorical consequence composition"],
        ["Location", "tract, CollisionsAll, CollisionsPUDO", "raw spatial concentration; no tract exposure"],
    ],
    columns=["grain", "features", "role"],
)
feature_map


## 2. Preprocessing order

1. Filter to Waymo driverless records using TCPID and source path.
2. Parse the 2024 aggregate schema separately from the 2025+ row schema.
3. Remove thousands separators before numeric conversion and fail on invalid
   critical exposure fields.
4. Preserve `Y=True`, `N=False`, and redacted/blank as `null`; unknown is not
   zero.
5. Stream-filter structural all-N template rows so they do not become fake
   independent observations.
6. Use `Incidents-Location.CollisionsPUDO` for 2025+ collision frequency;
   retain incident rows for actor/severity linkage and show the difference.
7. Build separate monthly exposure, event, location, and window-summary tables.
8. Engineer ratios/shares only after validating denominators and period
   coverage.

No SMOTE, train/test split, winsorization, or imputation is appropriate here:
this is a tiny longitudinal rate analysis, not an individual-level classifier.


## 3. Operational features and collinearity

The monthly variables largely move together because the service is expanding.
They are useful context, but not independent predictors that can all be put
into a six-window regression.


In [ ]:
m = monthly.sort(["year", "month"]).to_pandas()
m["date"] = pd.to_datetime(dict(year=m["year"], month=m["month"], day=1))
m["month_index"] = np.arange(len(m))
m["vmt_per_trip"] = m["vmt_total"] / m["TotalTrips"]
m["passengers_per_trip"] = m["TotalPassengersCarried"] / m["TotalTrips"]
m["pmt_per_vmt"] = m["TotalPMT"] / m["vmt_total"]
m["waiting_per_trip"] = m["TotalWaiting"] / m["TotalTrips"]
for period in [1, 2, 3]:
    m[f"p{period}_share"] = m[f"TotalVMTPeriod{period}"] / m["vmt_total"]

numeric_features = [
    "month_index", "TotalTrips", "vmt_total", "TotalPassengersCarried",
    "TotalPMT", "TotalWaiting", "vmt_per_trip", "passengers_per_trip",
    "pmt_per_vmt", "waiting_per_trip", "p1_share", "p2_share", "p3_share",
]
correlations = m[numeric_features].corr(method="pearson")
zero_variance = [
    column for column in [
        "TotalWAVsAvail", "TotalWAVsRequested",
        "TotalWAVsDeclinedUnavail", "TotalWAVsFulfilled",
    ]
    if m[column].nunique(dropna=False) == 1
]
vmt_reconciliation = (m["TotalVMTZEV"] - m["vmt_total"]).abs().max()

quality_checks = pd.DataFrame(
    {
        "check": [
            "unique months", "non-positive exposure rows",
            "max |TotalVMTZEV - P1-P2-P3|", "zero-variance WAV fields",
        ],
        "result": [
            m["date"].nunique(),
            int(((m["TotalTrips"] <= 0) | (m["vmt_total"] <= 0)).sum()),
            vmt_reconciliation,
            ", ".join(zero_variance),
        ],
    }
)
display(quality_checks)
display(correlations.loc[["month_index", "TotalTrips", "vmt_total"], ["TotalTrips", "vmt_total", "p1_share", "p3_share"]])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for column, label in [("TotalTrips", "Trips"), ("vmt_total", "VMT"), ("TotalPassengersCarried", "Passengers")]:
    axes[0].plot(m["date"], m[column] / m[column].iloc[0], marker="o", label=label)
axes[0].set_title("Operating scale (first month = 1)")
axes[0].set_ylabel("Relative level")
axes[0].legend()

axes[1].stackplot(
    m["date"], m["p1_share"], m["p2_share"], m["p3_share"],
    labels=["P1 deadhead", "P2 en-route", "P3 passenger"], alpha=0.85,
)
axes[1].set_ylim(0, 1)
axes[1].set_title("VMT composition changes over time")
axes[1].set_ylabel("Share of total VMT")
axes[1].legend(loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
feature_decisions = pd.DataFrame(
    [
        ["time_months", "keep", "one primary trend predictor"],
        ["log(VMT)", "offset (primary)", "course-requested exposure control"],
        ["log(trips)", "offset (sensitivity)", "closer to PUDO opportunity"],
        ["VMT + trips together", "exclude", "r≈0.999 and n=6"],
        ["TotalVMTZEV", "exclude", "duplicates P1+P2+P3"],
        ["four WAV totals", "exclude from model", "zero variance in Waymo driverless row"],
        ["VMT/trip and P1/P2/P3 shares", "describe", "service mix changes; too few windows for multivariable model"],
        ["complaint rate as predictor", "exclude", "derived from the outcome; target leakage"],
    ],
    columns=["feature", "decision", "reason"],
)
feature_decisions


## 4. Hazard and consequence before likelihood

PUDO hazards include stopping away from a safe curb, conflicts with moving
vehicles/cyclists/pedestrians while a passenger enters or exits, obstruction,
routing/localization failure, and accessibility-related unsafe crossings.

The measurable outcomes differ:

- a PUDO complaint is an unsolicited service/safety signal, not proof of a
  collision or injury;
- a PUDO collision is an event in CPUC's PUDO time window, not proof of fault;
- severity flags are available for 2025+ incident rows, but 2024 all-collision
  severity cannot be linked specifically to PUDO events.


In [ ]:
post_trips = post["trips"].sum()
post_vmt = post["vmt"].sum()
consequence_counts = {
    "PUDO complaint": int(post["pudo_complaints"].sum()),
    "PUDO collision (location aggregate)": int(post["pudo_collisions"].sum()),
    "PUDO collision with any listed severity": int(post["pudo_collision_severity_any_positive"].sum()),
    "PUDO collision, minor-or-worse": int(post["pudo_collision_severity_minor_or_worse"].sum()),
    "PUDO collision, severe": int(post["pudo_collision_worst_injury_severe"].sum()),
    "PUDO collision, fatal": int(post["pudo_collision_worst_fatality"].sum()),
}
consequence_table = pd.DataFrame(
    [
        [label, count, count / post_trips * 100_000, count / post_vmt * 1_000_000]
        for label, count in consequence_counts.items()
    ],
    columns=["outcome", "count", "per 100k trips", "per million VMT"],
)
fatal_upper_95_per_100k_trips = -np.log(0.05) / post_trips * 100_000
display(consequence_table)
print(
    f"Observed fatalities = 0; one-sided 95% Poisson upper bound = "
    f"{fatal_upper_95_per_100k_trips:.4f} per 100k trips (under complete ascertainment)."
)


In [ ]:
severity_labels = [
    "No listed severity flag", "Property damage", "Possible injury",
    "Minor injury", "Severe injury", "Fatality",
]
severity_columns = [
    "pudo_collision_worst_no_listed_severity",
    "pudo_collision_worst_property_damage",
    "pudo_collision_worst_injury_possible",
    "pudo_collision_worst_injury_minor",
    "pudo_collision_worst_injury_severe",
    "pudo_collision_worst_fatality",
]
severity_counts = post[severity_columns].sum().astype(int)
severity_table = pd.DataFrame(
    {"highest listed severity": severity_labels, "unique actor rows": severity_counts.values}
)

actor_map = {
    "Passenger": "pudo_collision_passenger",
    "Pedestrian": "pudo_collision_pedestrian",
    "Bicycle/scooter": "pudo_collision_bicycle_scooter",
    "Motorcycle": "pudo_collision_motorcycles",
    "Other vehicle": "pudo_collision_other_vehicle",
    "Rail": "pudo_collision_rail",
    "Property": "pudo_collision_property",
}
actor_table = pd.DataFrame(
    {
        "actor/property flag": list(actor_map),
        "positive flags": [int(post[column].sum()) for column in actor_map.values()],
    }
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].bar(severity_table["highest listed severity"], severity_table["unique actor rows"], color="#b24a42")
axes[0].tick_params(axis="x", rotation=35)
axes[0].set_title("Highest listed severity per PUDO collision row")
axes[0].set_ylabel("Unique rows")
axes[1].barh(actor_table["actor/property flag"], actor_table["positive flags"], color="#4c78a8")
axes[1].set_title("PUDO collision flags (multi-label; do not sum)")
plt.tight_layout()
plt.show()
display(severity_table)
display(actor_table)


In [ ]:
modern_events = events.filter(
    pl.col("period_id").str.starts_with("2025")
    | pl.col("period_id").str.starts_with("2026")
)
modern_pudo_complaints = modern_events.filter(pl.col("ComplaintPUDO") == 1)
coflag_columns = [
    "ComplaintSafety", "ComplaintAccessibility", "ComplaintWAV",
    "ComplaintCustomerService", "ComplaintOther", "CollisionPUDOAny",
]
complaint_coflags = modern_pudo_complaints.select(
    [pl.col(column).fill_null(0).sum().alias(column) for column in coflag_columns]
).to_pandas()

reconciliation = s[[
    "period_id", "pudo_collisions_incident_report", "pudo_collisions_location",
    "pudo_collision_reconciliation_gap", "pudo_collision_primary_source",
]].copy()
display(complaint_coflags)
display(reconciliation)
print(
    "Travel-lane status estimable from location data:",
    not locations.get_column("PUDOTravelLane").is_null().all(),
)


Interpretation: modern PUDO complaint rows do not overlap PUDO collision rows
or the other published complaint categories in this extract. Therefore the
complaint series cannot be treated as a physical-harm series. Actor flags are
multi-label, severity flags are collapsed to one highest listed level per row,
and “no listed severity flag” is not renamed “no harm.” Travel-lane frequency
is unavailable because the location field is redacted, not zero.


## 5. Reporting-window and schema caveat

The class slide describes Q3 2024–Q2 2026 as eight quarters. CPUC published
Jun–Aug, Sep–Nov, and a one-month December stub in 2024, then six calendar
quarters in 2025–2026. Redacted complaint dates prevent reassignment to true
calendar quarters.

- All nine real windows are retained for transparent description.
- Six 2025+ windows are the primary comparable-schema model sample.
- For the requested four-versus-four Y1/Y2 exercise only, Sep–Nov and December
  2024 are combined into a labeled transition block.


In [ ]:
def combine_rows(label, rows, year_group):
    complaint_count = rows["pudo_complaints"].sum()
    collision_count = rows["pudo_collisions"].sum()
    vmt = rows["vmt"].sum()
    trips = rows["trips"].sum()
    return {
        "block": label,
        "year_group": year_group,
        "pudo_complaints": complaint_count,
        "pudo_collisions": collision_count,
        "vmt": vmt,
        "trips": trips,
        "pudo_per_100k_vmt": complaint_count / vmt * 100_000,
        "pudo_per_100k_trips": complaint_count / trips * 100_000,
    }

course_blocks = pd.DataFrame(
    [
        combine_rows("2024 Q3 report (Jun–Aug)", s.iloc[[0]], "Y1"),
        combine_rows("2024 Q4 transition (Sep–Dec)", s.iloc[1:3], "Y1"),
        combine_rows("2025 Q1", s.iloc[[3]], "Y1"),
        combine_rows("2025 Q2", s.iloc[[4]], "Y1"),
        combine_rows("2025 Q3", s.iloc[[5]], "Y2"),
        combine_rows("2025 Q4", s.iloc[[6]], "Y2"),
        combine_rows("2026 Q1", s.iloc[[7]], "Y2"),
        combine_rows("2026 Q2", s.iloc[[8]], "Y2"),
    ]
)
course_blocks


## 6. Distributions: counts conditional on exposure

The monthly values form a growing time series, not 25 independent draws from a
stationary distribution. The window counts are also conditional on changing
exposure. Accordingly, dot/time plots, exact Poisson rate intervals, median /
IQR, and exposure-offset count models are more informative than a histogram.


In [ ]:
def distribution_row(label, values):
    values = pd.Series(values).dropna()
    return {
        "series": label,
        "n": len(values),
        "mean": values.mean(),
        "median": values.median(),
        "IQR": values.quantile(0.75) - values.quantile(0.25),
        "CV": values.std(ddof=1) / values.mean(),
        "skew": stats.skew(values, bias=False),
        "Shapiro p (low power at small n)": stats.shapiro(values).pvalue,
    }

distribution_summary = pd.DataFrame(
    [
        distribution_row("All 9: complaints/100k VMT", s["pudo_per_100k_vmt"]),
        distribution_row("2025+: complaints/100k VMT", post["pudo_per_100k_vmt"]),
        distribution_row("2025+: complaints/100k trips", post["pudo_per_100k_trips"]),
        distribution_row(
            "2025+: collisions/100k VMT",
            post["pudo_collisions"] / post["vmt"] * 100_000,
        ),
    ]
)
distribution_summary


In [ ]:
def poisson_rate_interval(count, exposure, scale=100_000, alpha=0.05):
    lower_count = 0 if count == 0 else stats.chi2.ppf(alpha / 2, 2 * count) / 2
    upper_count = stats.chi2.ppf(1 - alpha / 2, 2 * (count + 1)) / 2
    return lower_count / exposure * scale, upper_count / exposure * scale

post_rate = post.copy()
post_rate["complaint_rate"] = post_rate["pudo_complaints"] / post_rate["vmt"] * 100_000
intervals = [
    poisson_rate_interval(count, exposure)
    for count, exposure in zip(post_rate["pudo_complaints"], post_rate["vmt"])
]
post_rate["rate_low"], post_rate["rate_high"] = np.array(intervals).T

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].bar(s["period_id"], s["pudo_complaints"], color="#b54a3a")
axes[0].set_title("Raw complaints grow with service")
axes[0].set_ylabel("PUDO complaint count")
axes[0].tick_params(axis="x", rotation=55)

yerr = np.vstack([
    post_rate["complaint_rate"] - post_rate["rate_low"],
    post_rate["rate_high"] - post_rate["complaint_rate"],
])
axes[1].errorbar(
    post_rate["period_id"], post_rate["complaint_rate"], yerr=yerr,
    fmt="o-", capsize=4, color="#2f6f9f",
)
axes[1].set_title("Comparable-schema rate with exact 95% CI")
axes[1].set_ylabel("Complaints per 100k VMT")
axes[1].tick_params(axis="x", rotation=45)

for values, label in [
    (s["pudo_per_100k_vmt"], "All 9 (schema-mixed)"),
    (post["pudo_per_100k_vmt"], "2025+ comparable"),
]:
    ordered = np.sort(values)
    axes[2].step(ordered, np.arange(1, len(ordered) + 1) / len(ordered), where="post", label=label)
axes[2].set_title("Empirical CDF of complaint/VMT rate")
axes[2].set_xlabel("Complaints per 100k VMT")
axes[2].set_ylabel("Cumulative share")
axes[2].legend()
plt.tight_layout()
plt.show()


In [ ]:
complaint_categories = {
    "Safety": "complaint_safety",
    "PUDO": "pudo_complaints",
    "Other": "complaint_other",
    "Customer service": "complaint_customer_service",
    "Accessibility": "complaint_accessibility",
    "WAV": "complaint_wav",
}
category_counts = post[["period_id", *complaint_categories.values()]].set_index("period_id")
category_counts = category_counts.rename(columns={v: k for k, v in complaint_categories.items()})
category_shares = category_counts.div(category_counts.sum(axis=1), axis=0)

actor_counts = pd.Series(
    {label: int(post[column].sum()) for label, column in actor_map.items()},
    name="positive flags",
).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
category_shares.plot(kind="bar", stacked=True, ax=axes[0], colormap="tab20c")
axes[0].set_title("Complaint category composition (mutually exclusive)")
axes[0].set_ylabel("Share of published complaint rows")
axes[0].legend(loc="upper left", bbox_to_anchor=(1.01, 1))
actor_counts.sort_values().plot(kind="barh", ax=axes[1], color="#4c78a8")
axes[1].set_title("PUDO collision actors/properties (multi-label)")
axes[1].set_xlabel("Positive flags; totals overlap")
plt.tight_layout()
plt.show()


## 7. Course-requested Y1 versus Y2 tests

Welch's test and Mann–Whitney are reported because the lecture asks for a
parametric and a non-parametric comparison. With four blocks per group, power
is low. More importantly, Y1 mixes the 2024 aggregate and 2025 row schemas, so
the normalized-rate tests are descriptive checks—not causal evidence.


In [ ]:
test_rows = []
for metric in ["vmt", "pudo_complaints", "pudo_per_100k_vmt", "pudo_per_100k_trips"]:
    y1 = course_blocks.loc[course_blocks["year_group"] == "Y1", metric]
    y2 = course_blocks.loc[course_blocks["year_group"] == "Y2", metric]
    welch = stats.ttest_ind(y1, y2, equal_var=False)
    mann = stats.mannwhitneyu(y1, y2, alternative="two-sided")
    test_rows.append(
        {
            "metric": metric,
            "Y1 mean": y1.mean(),
            "Y2 mean": y2.mean(),
            "Welch p": welch.pvalue,
            "Mann–Whitney p": mann.pvalue,
        }
    )
pd.DataFrame(test_rows)


## 8. Trend models

The model is a Poisson count regression with actual window midpoint as the
single predictor and `log(exposure)` as an offset. VMT is primary; trips are a
mechanism-based sensitivity analysis. With only six modern windows, HC0 robust
standard errors can become spuriously narrow under underdispersion. We instead
use model-based Poisson uncertainty and inflate it only when Pearson dispersion
exceeds 1 (conservative quasi-Poisson rule).


In [ ]:
def fit_count_trend(data, outcome, exposure, label):
    X = sm.add_constant(data[["time_months"]])
    model = sm.GLM(
        data[outcome],
        X,
        family=sm.families.Poisson(),
        offset=np.log(data[exposure]),
    ).fit()
    dispersion = model.pearson_chi2 / model.df_resid
    scale = max(1.0, dispersion)
    beta = model.params["time_months"]
    standard_error = model.bse["time_months"] * np.sqrt(scale)
    low, high = beta + np.array([-1, 1]) * 1.96 * standard_error
    z_value = beta / standard_error
    return {
        "model": label,
        "outcome": outcome,
        "exposure": exposure,
        "n windows": len(data),
        "annual change": np.exp(beta * 12) - 1,
        "95% CI low": np.exp(low * 12) - 1,
        "95% CI high": np.exp(high * 12) - 1,
        "p": 2 * stats.norm.sf(abs(z_value)),
        "Pearson dispersion": dispersion,
        "SE scale used": scale,
    }

primary_models = pd.DataFrame(
    [
        fit_count_trend(post, "pudo_complaints", "vmt", "2025+ complaint"),
        fit_count_trend(post, "pudo_complaints", "trips", "2025+ complaint"),
        fit_count_trend(post, "pudo_collisions", "vmt", "2025+ collision"),
        fit_count_trend(post, "pudo_collisions", "trips", "2025+ collision"),
    ]
)
full_window_sensitivity = pd.DataFrame(
    [
        fit_count_trend(s, "pudo_complaints", "vmt", "All 9; schema-confounded"),
        fit_count_trend(s, "pudo_complaints", "trips", "All 9; schema-confounded"),
    ]
)
display(primary_models)
display(full_window_sensitivity)


In [ ]:
leave_one_out = []
for outcome in ["pudo_complaints", "pudo_collisions"]:
    for exposure in ["vmt", "trips"]:
        changes = []
        for index in post.index:
            row = fit_count_trend(
                post.drop(index), outcome, exposure, f"drop {post.loc[index, 'period_id']}"
            )
            changes.append(row["annual change"])
        leave_one_out.append(
            {
                "outcome": outcome,
                "exposure": exposure,
                "min annual slope": min(changes),
                "max annual slope": max(changes),
            }
        )
pd.DataFrame(leave_one_out)


In [ ]:
def poisson_rate_ratio(first, second, outcome, exposure):
    first_count, second_count = first[outcome].sum(), second[outcome].sum()
    first_exp, second_exp = first[exposure].sum(), second[exposure].sum()
    ratio = (second_count / second_exp) / (first_count / first_exp)
    standard_error = np.sqrt(1 / first_count + 1 / second_count)
    low, high = np.exp(np.log(ratio) + np.array([-1, 1]) * 1.96 * standard_error)
    p_value = 2 * stats.norm.sf(abs(np.log(ratio) / standard_error))
    return ratio, low, high, p_value

halves = {
    "2025 H1": post.iloc[0:2],
    "2025 H2": post.iloc[2:4],
    "2026 H1": post.iloc[4:6],
}
ratio_rows = []
for first_label, second_label in [("2025 H1", "2025 H2"), ("2025 H2", "2026 H1")]:
    for exposure in ["vmt", "trips"]:
        ratio, low, high, p_value = poisson_rate_ratio(
            halves[first_label], halves[second_label], "pudo_complaints", exposure
        )
        ratio_rows.append(
            {
                "comparison": f"{second_label} / {first_label}",
                "exposure": exposure,
                "rate ratio": ratio,
                "95% CI low": low,
                "95% CI high": high,
                "p": p_value,
            }
        )
pd.DataFrame(ratio_rows)


The modern complaint models should drive the wording. A point estimate alone
does not establish direction when its interval spans both meaningful decline
and meaningful increase. The collision model provides a downward signal, but
its conservative interval also crosses zero change. The full nine-window
model remains a sensitivity result because it cannot separate reporting-schema
change from risk change.


## 9. Scenario-specific provisional risk matrix

The lecture supplies qualitative likelihood labels but no numeric rate
cutoffs. To make placement reproducible, the following **project-defined**
bins use events per 100,000 completed trips:

- Rare: `<0.01`; Unlikely: `0.01–<0.1`; Moderate: `0.1–<1`;
  Likely: `1–<5`; Almost Certain: `≥5`.

These cutoffs must be replaced if the instructor/team adopts another rubric.
Separate scenarios are placed because a complaint, an injury-coded collision,
and a severe injury do not share one consequence.


In [ ]:
def likelihood_from_rate(rate):
    if rate < 0.01:
        return "Rare", 1
    if rate < 0.1:
        return "Unlikely", 2
    if rate < 1:
        return "Moderate", 3
    if rate < 5:
        return "Likely", 4
    return "Almost Certain", 5

scenario_inputs = [
    ("PUDO complaint/service signal", consequence_counts["PUDO complaint"], "Minor (provisional)", 2),
    (
        "PUDO collision: possible/minor injury",
        int(post["pudo_collision_worst_injury_possible"].sum() + post["pudo_collision_worst_injury_minor"].sum()),
        "Significant", 3,
    ),
    ("PUDO collision: severe injury", consequence_counts["PUDO collision, severe"], "Major", 4),
]
risk_rows = []
for scenario, count, consequence, consequence_score in scenario_inputs:
    rate = count / post_trips * 100_000
    likelihood, likelihood_score = likelihood_from_rate(rate)
    risk_rows.append(
        {
            "scenario": scenario,
            "count": count,
            "rate/100k trips": rate,
            "likelihood": likelihood,
            "likelihood score": likelihood_score,
            "consequence": consequence,
            "consequence score": consequence_score,
            "matrix score": likelihood_score * consequence_score,
        }
    )
risk_register = pd.DataFrame(risk_rows)
risk_register


In [ ]:
likelihood_labels = ["Rare 1", "Unlikely 2", "Moderate 3", "Likely 4", "Almost Certain 5"]
impact_labels = ["Insignificant 1", "Minor 2", "Significant 3", "Major 4", "Severe 5"]
scores = np.outer(np.arange(1, 6), np.arange(1, 6))
categories = np.select(
    [scores <= 2, scores <= 4, scores <= 9, scores <= 12, scores <= 16],
    [0, 1, 2, 3, 4],
    default=5,
)
cmap = ListedColormap(["#4f9d69", "#8ac46b", "#f2d45c", "#f39c4a", "#df6258", "#a83232"])

fig, ax = plt.subplots(figsize=(10.5, 6))
ax.imshow(categories, cmap=cmap, vmin=0, vmax=5, origin="lower")
for row in range(5):
    for column in range(5):
        ax.text(column, row, str(scores[row, column]), ha="center", va="center", fontsize=10)
markers = ["o", "s", "^"]
for marker, (_, scenario) in zip(markers, risk_register.iterrows()):
    ax.scatter(
        scenario["consequence score"] - 1,
        scenario["likelihood score"] - 1,
        marker=marker,
        s=180,
        facecolors="none",
        edgecolors="black",
        linewidths=2,
        label=scenario["scenario"],
    )
ax.set_xticks(range(5), impact_labels, rotation=20, ha="right")
ax.set_yticks(range(5), likelihood_labels)
ax.set_xlabel("Consequence")
ax.set_ylabel("Likelihood (project-defined rate bins)")
ax.set_title("Scenario-specific PUDO risk matrix")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()


## 10. Conclusion

- **Samples/features:** operational scale variables are strongly time-trended
  and collinear. Use one time predictor and one exposure offset; do not treat
  raw event/template rows as independent model samples.
- **Preprocessing:** schema-version parsing, nullable redaction, template-row
  filtering, exact exposure coverage, mutually exclusive highest severity, and
  location/incident reconciliation are required before modeling.
- **Distribution:** the full nine-window complaint rate is skewed largely by
  the 2024 schema/level shift. The six comparable modern rates occupy a much
  narrower range, but six windows cannot establish a stable population
  distribution.
- **Likelihood trend:** report annual slope, interval, and p-value from the
  2025+ offset models. Current evidence does not establish that complaint rate
  is rising or falling; collision rate has a downward signal but remains
  statistically uncertain.
- **Consequence:** 2025+ contains 318 location-aggregated PUDO collisions. Of
  315 actor-linked rows, 23 have a positive severity flag, 12 are
  minor-or-worse, one is severe, and none is fatality-coded. Zero observed
  fatalities is not zero future risk.
- **Risk decision:** keep service complaints, physical collisions, and injury
  severity as separate scenarios. Matrix placement is provisional until the
  team/instructor approves likelihood thresholds and consequence rules.
